In [1]:
!python -m pip install -e '.[test]'
!python -m neuropath2path --help

Obtaining file:///home/timonidis/PycharmProjects/NeuroPath2Path/notebooks

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
ERROR: file:///home/timonidis/PycharmProjects/NeuroPath2Path/notebooks does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
usage: python -m neuropath2path [-h] [--version]
                                {features,compare,extract,classify,weights} ...

Command-line entry points for all workflows present in the MATLAB sources.

positional arguments:
  {features,compare,extract,classify,weights}
    features            Extract all path features from an SWC
    compare             Compare two SWC neuronal arbors
    extract             Summarize/extract a directory or ZIP dataset
    classify            Run interclass/intraclass evaluation
    weights             Learn importance weights for every class pair

options:
  -h, --help            show this help message and ex

#### Classification

#### Original interclass experiment settings

In [ ]:
!python -m neuropath2path classify ../Intercellular_data.zip \
    --mode inter --test-proportion 0.1 --bin-width 15 \
    --search-step 5 --neighbors 5 --search-span 9 \
    --seed 100 --jobs 6 --output results/inter.json

#### Original intraclass experiment settings:

In [ ]:
!python -m neuropath2path classify ../Intracellular_data.zip \
  --mode intra --test-proportion 0.1 --bin-width 7 \
  --search-step 5 --neighbors 5 --search-span 10 \
  --seed 100 --jobs 6 --output results/intra.json

#### Quick check on a small, deterministic subset:

In [ ]:
!python -m neuropath2path classify ../Intercellular_data.zip \
  --limit-per-class 4 --test-proportion 0.5 --bin-width 100000 \
  --neighbors 1 --seed 100 --output results/smoke.json

##### The subset takes the first N dendritic neurons per class in sorted file order. Its accuracy is a smoke check, not an estimate of full-dataset performance. The large bin width keeps each small class in a single split bin.

Each class is split into path-count bins. Training uses max(1, floor((1 - test_proportion) * bin_count)) neurons from each bin, so singleton bins stay in training. Actual test proportions can differ from the requested fraction. Candidate search expands a strict path-count window separately for each class. If fewer than k candidates are found in total, all training neurons are considered and k is capped at the training set size. Test neurons never enter the candidate pool.

JSON output includes class order, the confusion matrix (true labels in rows), accuracy as a fraction, predictions, neighbor distances and sources, fallback flags, and train/test indices into record_sources. Parameters and seed are saved. Progress goes to stderr; omit --output to emit JSON to stdout, or add --quiet to suppress progress.

Full experiments, especially feature-weight learning, can be expensive: each candidate comparison evaluates all pairs of dendritic paths. --jobs controls concurrent test-neuron evaluations. On Slurm, activate the installed environment and submit from the repository root:

In [ ]:
!sbatch scripts/classify_inter.slurm
!sbatch scripts/classify_intra.slurm
!sbatch scripts/learn_weights.sl

#### Features and comparison

In [ ]:
!python -m neuropath2path features neuron.swc --output results/features.json
!python -m neuropath2path compare first.swc second.swc \
    --mode inter --output results/comparison.json
!python -m neuropath2path extract Intercellular_data.zip \
    --output results/dataset-summary.json

`features` exports all per-path arrays. `extract` reports path counts, average
lengths, and maximum orders; add `--include-paths` for all arrays. Tip types
3 and 4 define dendritic paths. Axon-only neurons are recorded and skipped
when loading a dataset.

Each path stores tortuosity, bifurcation angle (radians), partition asymmetry,
concurrence, hierarchy, segment length, competition, and original SWC node IDs.
The six distance features, in coefficient order, are tortuosity, bifurcation
angle, partition asymmetry, concurrence, segment length, and competition.
Hierarchy supplies segment weights.

`compare` returns a scalar distance, a 6-by-number-of-matches feature-distance
matrix, zero-based path pairs, and the path cost matrix. Original SWC node IDs
remain unchanged; they are distinct from zero-based path indices.

| Mode | Alignment | Path cost |
| --- | --- | --- |
| `inter` | Pad at the tip | Sigmoid of a weighted sum |
| `intra` | Pad at the root | Weighted sum |
| `raw` | Pad at the tip | Euclidean norm, for learning weights |

Interclass costs have a baseline of **0.5**, including identical neurons with
the corrected calculation. Intraclass and raw identical-neuron costs are zero.
Matching repeatedly assigns the smaller arbor to unused paths in the larger
arbor, assigns the remainder, then applies the original skewness-based outlier
reassignment. These scores are not guaranteed to be mathematical metrics.

In [ ]:
from neuropath2path import read_swc, extract_features, compute_distance

first = extract_features(read_swc("first.swc"))
second = extract_features(read_swc("second.swc"))
result = compute_distance(first, second, mode="inter")
print(result.distance, result.pairs)

# Three-return unpacking is also supported:
distance, feature_vectors, path_pairs = result

#### Feature importance

In [ ]:
!python -m neuropath2path weights Intercellular_data.zip \
  --limit-per-class 3 --seed 100 --output results/weights
